**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `python_prebuilt/` and the bundle's R routes (`R/lib`, see `R/README_BRIDGE.md`; the full R pipeline is the project RWDR). Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# V01 — Analytical Formula Verification
**The strongest form of proof available**: these check the estimators against *exact closed-form
solutions*, not simulations with sampling noise. A mismatch here is a genuine formula error.

| Check | Expected |
|---|---|
| TWFE == textbook 2x2 DiD identity | agreement to machine precision (~1e-15) |
| Cluster-robust SE == Cameron-Gelbach-Miller CR1 | exact match to a hand-computed value |
| Callaway-Sant'Anna collapses to 2x2 (single cohort) | difference exactly 0.0 |

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (v20.55: the engine installs a missing ESTIMATOR package itself -- wheel first, then source -- and confirms 'N of N installed' at every run; these base libraries it needs before it can start)")


In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")

sys.path.insert(0,"../All_DIDsCodes"); sys.path.insert(0,"All_DIDsCodes")
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.44")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("V01")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
C.autotune()          # v17: measures THIS machine (cores/RAM/VRAM/disk) and sets every performance knob
C.step(1,"TWFE must EQUAL the closed-form 2x2 DiD identity")
rng=np.random.default_rng(99); n=37; data=[]; means={}
for tr in [0,1]:
    for po in [0,1]:
        v=rng.normal(10+2*tr+3*po+5*tr*po,2,n); means[(tr,po)]=v.mean()
        data+= [{"y":x,"treat":tr,"post":po} for x in v]
d=pd.DataFrame(data); d["did_term"]=d.treat*d.post
beta,_=C.estimate_twfe_did(d,"y","did_term","treat","post","treat")
closed=(means[(1,1)]-means[(1,0)])-(means[(0,1)]-means[(0,0)])
diff=abs(beta-closed)
print(f"closed-form = {closed:.12f}\nTWFE        = {beta:.12f}\ndifference  = {diff:.3e}")
(C.ok if diff<1e-9 else C.fail)(f"TWFE == closed-form 2x2 identity (diff {diff:.2e})")
assert diff<1e-9
C.done(1, next_task="next cell")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
C.step(2,"cluster-robust SE must match the published CR1 formula")
X=np.array([[1.],[2.],[3.],[1.5],[2.5],[3.5]]); r=np.array([.5,-.3,.2,-.1,.4,-.6])
cl=np.array(["A","A","A","B","B","B"])
v=C.cluster_robust_se(X,r,cl)
n,k,G=6,1,2; dfc=(G/(G-1))*((n-1)/(n-k))
XtXi=np.linalg.inv(X.T@X); meat=np.zeros((1,1))
for c in ["A","B"]:
    s=X[cl==c].T@r[cl==c]; meat+=np.outer(s,s)
exp=dfc*XtXi@meat@XtXi
print(f"dfc = (G/(G-1))*((N-1)/(N-K)) = {dfc}")
(C.ok if np.allclose(v,exp) else C.fail)(f"matches hand-computed CR1: {np.allclose(v,exp)}")
assert np.allclose(v,exp)
C.done(2, next_task="next cell")

In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
C.step(3,"Callaway-Sant'Anna must collapse EXACTLY to 2x2 in the single-cohort case")
rng=np.random.default_rng(7); n=50; rows=[]; mm={}
for tr in [0,1]:
    for t in [2021,2022]:
        v=rng.normal(5+2*tr+1*(t==2022)+3*tr*(t==2022),1.5,n); mm[(tr,t)]=v.mean()
        rows+=[{"unit":f"{tr}_{i}","time":t,"y":x,"first_treat":2022 if tr else np.inf}
               for i,x in enumerate(v)]
d=pd.DataFrame(rows)
tbl,_=C.callaway_santanna_att(d,"y","unit","time","first_treat")
cs=tbl["ATT_gt"].iloc[0]; cf=(mm[(1,2022)]-mm[(1,2021)])-(mm[(0,2022)]-mm[(0,2021)])
print(f"closed-form 2x2 = {cf:.12f}\nCallaway-Sant'Anna = {cs:.12f}\ndiff = {abs(cs-cf):.3e}")
(C.ok if abs(cs-cf)<1e-9 else C.fail)("CS collapses exactly to the 2x2 identity")
assert abs(cs-cf)<1e-9
C.ok("V01 COMPLETE -- all analytical identities verified")

# ---- MEMORY RELEASE (v14): free this model's frames so the next one starts clean ----
try:
    C.release(*[v for k,v in list(globals().items()) if k in ("panel","df","result") and v is not None])
except Exception as _e:
    print(f"[INFO]    release skipped: {_e}")
C.done(3, next_task="run V02_Fixed_Effects_Consistency_Check.ipynb")